<a href="https://colab.research.google.com/github/PovSobek/Manipuladores/blob/main/07_Jacobiano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='steelblue'> Introducción al Jacobiano en Robótica </font>

**Material desarrollado por Vicente Esteve-Sala**

![](https://drive.google.com/thumbnail?id=1GrzpXe8zsvQwyY7zTw87VCG4c14sHTvt&sz=w800)

**Fecha última edición**: 13/10/2025

**Licencia**:
<small> © 2025 by <a href="https://cvnet.cpd.ua.es/curriculum-breve/es/esteve-sala-vicente-manuel/283903">Vicente Esteve-Sala</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0       </a><small><a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br /></small>

Al usar estos contenidos, aceptas los términos de uso, propiedad intelectual y la política de privacidad de la UA.

# El Jacobiano - Controlando la Velocidad del Robot ⚡

Hasta ahora hemos controlado la **posición** del robot. El siguiente nivel es controlar su **velocidad**. Queremos poder decir: "muévete hacia adelante a 0.5 m/s". Aquí es donde entra el Jacobiano.

El **Jacobiano (J)** es una matriz que relaciona las velocidades de las articulaciones (velocidad angular $\dot{q}$) con la velocidad del efector final (velocidad cartesiana $\dot{x}$).

La relación fundamental es:
$$ \dot{x} = J(q) \dot{q} $$
<br><br>
O en detalle para nuestro robot 2D:
$$ \begin{bmatrix} v_x \\ v_y \end{bmatrix} = J(\theta_1, \theta_2) \begin{bmatrix} \dot{\theta_1} \\ \dot{\theta_2} \end{bmatrix} $$
<br>
* $\dot{x} = [v_x, v_y]$: La velocidad de la pinza en las direcciones X e Y.

* $\dot{q} = [\dot{\theta_1}, \dot{\theta_2}]$: Las velocidades angulares de los motores 1 y 2.

* $J(q)$: El Jacobiano, que depende de la configuración actual del robot $(\theta_1, \theta_2)$.<br><br>

En resumen, el Jacobiano nos permite traducir las velocidades de los motores a la velocidad de la pinza, y viceversa.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Entrada de Datos Interactiva ---
try:
    print("--- Introduce los PARÁMETROS del ROBOT (2 Eslabones) ---")
    L1 = float(input("Longitud del eslabón 1 (L1) en metros: "))
    L2 = float(input("Longitud del eslabón 2 (L2) en metros: "))

    print("\n--- Introduce la CONFIGURACIÓN del Robot ---")
    theta1_grados = float(input("Ángulo de la articulación 1 (grados): "))
    theta2_grados = float(input("Ángulo de la articulación 2 (grados): "))

    print("\n¡Entorno preparado!")

except ValueError:
    print("\nError: Por favor, introduce solo valores numéricos. Vuelve a ejecutar la celda.")

## La Matriz Jacobiana

El Jacobiano se obtiene derivando las ecuaciones de la cinemática directa con respecto a cada ángulo. Para nuestro brazo de 2 eslabones, la matriz resultante es:

$$ J = \begin{bmatrix} \frac{\partial x}{\partial \theta_1} & \frac{\partial x}{\partial \theta_2} \\ \frac{\partial y}{\partial \theta_1} & \frac{\partial y}{\partial \theta_2} \end{bmatrix} = \begin{bmatrix} -L_1 s_1 - L_2 s_{12} & -L_2 s_{12} \\ L_1 c_1 + L_2 c_{12} & L_2 c_{12} \end{bmatrix} $$

Donde usamos una notación abreviada:
* $s_1 = \sin(\theta_1)$
* $c_1 = \cos(\theta_1)$
* $s_{12} = \sin(\theta_1 + \theta_2)$
* $c_{12} = \cos(\theta_1 + \theta_2)$

### Ejercicio Práctico:
Implementa una función en la siguiente celda que, dados los ángulos y longitudes, construya y devuelva esta matriz Jacobiana como un array de NumPy.

In [ ]:
def calcular_jacobiano(theta1_rad, theta2_rad, L1, L2):
    """
    Calcula la matriz Jacobiana para el robot de 2-DOF en una configuración dada.
    """
    # --- ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼ ---

    # Calcula los senos y cosenos necesarios para la matriz
    s1 = 0
    c1 = 0
    s12 = 0
    c12 = 0

    # Define los 4 elementos de la matriz
    j11 = 0
    j12 = 0
    j21 = 0
    j22 = 0

    # --- ▲▲▲ FIN DE TU CÓDIGO ▲▲▲ ---

    # Construye la matriz Jacobiana como un array de NumPy 2x2
    J = np.array([[j11, j12],
                  [j21, j22]])


    return J

# --- Probamos tu función ---
th1_rad = np.deg2rad(theta1_grados)
th2_rad = np.deg2rad(theta2_grados)
J_test = calcular_jacobiano(th1_rad, th2_rad, L1, L2)

print("La matriz Jacobiana para la configuración dada es:")
print(J_test)

## El Problema Inverso de Velocidad y las Singularidades

El uso más común del Jacobiano es resolver el problema inverso: "Si quiero que la pinza se mueva con una velocidad $\dot{x}$, ¿a qué velocidad $\dot{q}$ deben girar los motores?". Para ello, reorganizamos la ecuación:

$$ \dot{q} = J^{-1}(q) \dot{x} $$

Necesitamos calcular la **inversa del Jacobiano**. Pero, ¿qué pasa si la matriz no tiene inversa?

Esto ocurre en las **singularidades**: configuraciones del robot donde pierde la capacidad de moverse en ciertas direcciones. Para nuestro brazo, esto sucede cuando está completamente estirado ($\theta_2 = 0^\circ$) o completamente plegado sobre sí mismo ($\theta_2 = 180^\circ$). En esos puntos, el **determinante** del Jacobiano es cero y no se puede invertir.

In [ ]:
# --- CÁLCULO DEL PROBLEMA INVERSO DE VELOCIDAD ---
try:
    print("\n--- Introduce la VELOCIDAD DESEADA para la pinza ---")
    vx = float(input("Velocidad en X (vx) en m/s: "))
    vy = float(input("Velocidad en Y (vy) en m/s: "))
    v_deseada = np.array([vx, vy])

    # 1. Calcular el Jacobiano en la posición actual
    th1_rad = np.deg2rad(theta1_grados)
    th2_rad = np.deg2rad(theta2_grados)
    J = calcular_jacobiano(th1_rad, th2_rad, L1, L2)

    # 2. Comprobar si estamos en una singularidad
    determinante = np.linalg.det(J)
    print(f"\nDeterminante del Jacobiano: {determinante:.4f}")
    if np.isclose(determinante, 0):
        print("¡ADVERTENCIA! El robot está en una singularidad o muy cerca de una.")
        print("No se puede calcular una velocidad de motor única y finita.")
        q_dot = None
    else:
        # 3. Calcular la inversa y las velocidades de los motores
        J_inversa = np.linalg.inv(J)
        q_dot = J_inversa @ v_deseada # Multiplicación de matriz por vector

        print("\nPara alcanzar la velocidad deseada, los motores deben girar a:")
        print(f"  - Velocidad de Motor 1 (θ1_dot): {np.rad2deg(q_dot[0]):.2f} grados/s")
        print(f"  - Velocidad de Motor 2 (θ2_dot): {np.rad2deg(q_dot[1]):.2f} grados/s")

    # --- VISUALIZACIÓN ---
    # Posición del brazo (Cinemática Directa)
    x1 = L1 * np.cos(th1_rad)
    y1 = L1 * np.sin(th1_rad)
    x_pinza = x1 + L2 * np.cos(th1_rad + th2_rad)
    y_pinza = y1 + L2 * np.sin(th1_rad + th2_rad)

    fig, ax = plt.subplots(figsize=(8,8))
    ax.set_aspect('equal'); ax.grid(True)
    ax.set_xlim(-(L1+L2+1), L1+L2+1); ax.set_ylim(-(L1+L2+1), L1+L2+1)
    ax.set_title("Análisis de Velocidad Jacobiano")
    # Dibujar espacio de trabajo y brazo
    ax.add_patch(plt.Circle((0, 0), L1+L2, color='lightblue', alpha=0.5))
    ax.plot([0, x1], [0, y1], 'r-o', lw=3, ms=10, label='Eslabón 1')
    ax.plot([x1, x_pinza], [y1, y_pinza], 'b-o', lw=3, ms=10, label='Eslabón 2')

    # Dibujar el vector de velocidad deseado
    if q_dot is not None:
        ax.arrow(x_pinza, y_pinza, v_deseada[0], v_deseada[1], head_width=0.1,
                 head_length=0.15, fc='g', ec='g', label='Velocidad Deseada')
        # Anotar las velocidades de los motores
        ax.text(0.1, 0, f"θ1_dot: {np.rad2deg(q_dot[0]):.1f}°/s", fontsize=12, color='red',
                bbox=dict(facecolor='white', alpha=0.8))
        ax.text(x1, y1, f"θ2_dot: {np.rad2deg(q_dot[1]):.1f}°/s", fontsize=12, color='blue',
                bbox=dict(facecolor='white', alpha=0.8))

    ax.legend(fontsize='small'); plt.show()

except ValueError:
    print("\nError: Por favor, introduce solo valores numéricos. Vuelve a ejecutar la celda.")
except NameError:
    print("\nError: Parece que no has ejecutado la celda de configuración inicial.")

**Tu Tarea:**
1. Introduce los parámetros del robot.
2. Completa le código que falta en la celda del ejercicio práctico
3. Ejecuta las celdas con el código resuelto
4. Entrega en un pdf las capturas de pantallas y el código generado